In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import copy
# Import natywnego zbioru z torchvision
from torchvision.datasets import Imagenette

# Ustawienie urządzenia (GPU jeśli dostępne)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

# Hiperparametry
BATCH_SIZE = 64
EPOCHS_VOG = 5       # Liczba epok do wyliczenia VoG (rozgrzewka)
EPOCHS_TRAIN = 5     # Liczba epok do docelowego treningu
NUM_CLASSES = 10     # Imagenette ma 10 klas
SUBSET_FRACTION = 0.3 # Użyjemy 30% najtrudniejszych danych (wysokie VoG)

Używane urządzenie: cpu


In [2]:
# Wczytywanie i wrapper dla Datasetu
weights = ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()

class VoGDatasetWrapper(Dataset):
    """
    Wrapper, który zwraca obraz, etykietę oraz INDEKS próbki.
    Indeks jest nam niezbędny do śledzenia wariancji gradientów (VoG)
    dla konkretnych zdjęć w kolejnych epokach.
    """
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        # Pobranie danych bazowych (PIL Image, label)
        image, label = self.base_dataset[idx]

        # Zapewnienie formatu RGB (dla bezpieczeństwa)
        if image.mode != 'RGB':
            image = image.convert('RGB')

        # Aplikacja transformacji ResNet50
        image = self.transform(image)
        return image, label, idx

# UWAGA: download=True pobierze dataset automatycznie w katalogu './data'
print("Pobieranie/ładowanie datasetu Imagenette...")
train_base = Imagenette(root='./data', split='train', size='320px', download=True, transform=None)
val_base = Imagenette(root='./data', split='val', size='320px', download=True, transform=None)

train_dataset = VoGDatasetWrapper(train_base, preprocess)
val_dataset = VoGDatasetWrapper(val_base, preprocess)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Rozmiar zbioru treningowego: {len(train_dataset)}")
print(f"Rozmiar zbioru walidacyjnego: {len(val_dataset)}")

Pobieranie/ładowanie datasetu Imagenette...
Rozmiar zbioru treningowego: 9469
Rozmiar zbioru walidacyjnego: 3925


In [3]:
def get_model(frozen=True):
    """
    Tworzy model ResNet50.
    Jeśli frozen=True (Linear Probe), zamraża wagi i uczy tylko ostatnią warstwę.
    Jeśli frozen=False (Fine-tuning), uczy całą sieć.
    """
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    if frozen:
        for param in model.parameters():
            param.requires_grad = False

    # Zastąpienie ostatniej warstwy (Linear Probe zawsze uczy tę warstwę)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)

    return model.to(device)

In [ ]:
def calculate_vog(train_loader, epochs=EPOCHS_VOG):
    print("Rozpoczynam obliczanie Variance of Gradients (VoG)...")
    model = get_model(frozen=False) # Używamy odmrożonego modelu do śledzenia dynamiki
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Słownik do przechowywania gradientów dla każdego indeksu obrazka z przestrzeni epok
    gradients_history = {i: [] for i in range(len(train_loader.dataset))}

    model.train()
    for epoch in range(epochs):
        print(f"VoG Epoka {epoch+1}/{epochs}")
        for inputs, labels, indices in tqdm(train_loader, leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            # Wymagamy gradientów dla wejścia, by obliczyć "ważność" obrazka
            inputs.requires_grad_(True)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            # Zbieramy normę L2 gradientu dla każdego obrazka w batchu
            grad_norms = torch.norm(inputs.grad.view(inputs.size(0), -1), dim=1).detach().cpu().numpy()

            for idx, grad_norm in zip(indices.numpy(), grad_norms):
                gradients_history[idx].append(grad_norm)

            optimizer.step()

    # Obliczanie wariancji gradientów (VoG) dla każdej próbki
    vog_scores = {}
    for idx, grads in gradients_history.items():
        vog_scores[idx] = np.var(grads)

    print("Zakończono wyliczanie VoG.")
    return vog_scores

# Wykonaj obliczenia VoG
vog_scores_dict = calculate_vog(train_loader)

Rozpoczynam obliczanie Variance of Gradients (VoG)...
VoG Epoka 1/5


In [ ]:
# Sortujemy indeksy na podstawie wyników VoG (malejąco i rosnąco)
# High VoG = duża wariancja = trudne przykłady
sorted_indices_desc = sorted(vog_scores_dict.keys(), key=lambda k: vog_scores_dict[k], reverse=True)
# Low VoG = mała wariancja = łatwe / stabilne przykłady
sorted_indices_asc = sorted(vog_scores_dict.keys(), key=lambda k: vog_scores_dict[k], reverse=False)

# Wybieramy top 30% i bottom 30% próbek
subset_size = int(len(train_dataset) * SUBSET_FRACTION)

high_vog_indices = sorted_indices_desc[:subset_size]
low_vog_indices = sorted_indices_asc[:subset_size]

# Tworzymy nowe Dataloadery dla obu podzbiorów
# Pamiętajmy o num_workers=0, żeby uniknąć błędu multiprocessing
high_vog_subset = Subset(train_dataset, high_vog_indices)
high_vog_loader = DataLoader(high_vog_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

low_vog_subset = Subset(train_dataset, low_vog_indices)
low_vog_loader = DataLoader(low_vog_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"Wybrano {len(high_vog_subset)} próbek do zbioru HIGH VoG (Trudne).")
print(f"Wybrano {len(low_vog_subset)} próbek do zbioru LOW VoG (Łatwe).")

Wybrano 2840 próbek do zbioru HIGH VoG (Trudne).
Wybrano 2840 próbek do zbioru LOW VoG (Łatwe).


In [ ]:
def train_and_evaluate(model, train_dl, val_dl, epochs=EPOCHS_TRAIN, title="Model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    best_acc = 0.0
    history = {'train_loss': [], 'val_acc': []} # Słownik do logowania wyników

    print(f"\nRozpoczynam eksperyment: {title}")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        train_pbar = tqdm(train_dl, desc=f"[{title}] Epoka {epoch+1}/{epochs} [Trening]", leave=False)
        for inputs, labels, _ in train_pbar:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        # Zapis historii ze zbioru treningowego
        avg_train_loss = running_loss / len(train_dl)
        history['train_loss'].append(avg_train_loss)

        model.eval()
        correct = 0
        total = 0
        val_pbar = tqdm(val_dl, desc=f"[{title}] Epoka {epoch+1}/{epochs} [Walidacja]", leave=False)
        with torch.no_grad():
            for inputs, labels, _ in val_pbar:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        # Zapis historii ze zbioru walidacyjnego
        val_acc = 100 * correct / total
        history['val_acc'].append(val_acc)

        print(f"[{title}] Epoka {epoch+1}/{epochs} | Strata Treningowa: {avg_train_loss:.4f} | Dokładność Walidacji: {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc

    return history, best_acc

In [ ]:
histories = {}
best_accuracies = {}

# Słownik definiujący nasze 3 zbiory do testów
datasets_to_test = {
    "Full_Dataset": train_loader,
    "High_VoG_Subset": high_vog_loader,
    "Low_VoG_Subset": low_vog_loader
}

# Eksperyment 1: Zamrożony model (Linear Probe)
for ds_name, loader in datasets_to_test.items():
    title = f"Frozen_{ds_name}"
    model = get_model(frozen=True)
    hist, best_acc = train_and_evaluate(model, loader, val_loader, title=title)
    histories[title] = hist
    best_accuracies[title] = best_acc

# Eksperyment 2: Odmrożony model (Fine-Tuning)
for ds_name, loader in datasets_to_test.items():
    title = f"Unfrozen_{ds_name}"
    model = get_model(frozen=False)
    hist, best_acc = train_and_evaluate(model, loader, val_loader, title=title)
    histories[title] = hist
    best_accuracies[title] = best_acc

print("\n=== PODSUMOWANIE DOKŁADNOŚCI ===")
for experiment, acc in best_accuracies.items():
    print(f"{experiment}: {acc:.2f}%")


Rozpoczynam eksperyment: Frozen_Full_Dataset


[Frozen_Full_Dataset] Epoka 1/5 [Trening]:   0%|          | 0/148 [00:00<?, ?it/s]

[Frozen_Full_Dataset] Epoka 1/5 [Walidacja]:   0%|          | 0/62 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79eb50058a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79eb50058a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[Frozen_Full_Dataset] Epoka 1/5 | Strata Treningowa: 0.4385 | Dokładność Walidacji: 99.29%


...

[Unfrozen_Low_VoG_Subset] Epoka 4/5 [Walidacja]:   0%|          | 0/62 [00:00<?, ?it/s]

[Unfrozen_Low_VoG_Subset] Epoka 4/5 | Strata Treningowa: 0.0980 | Dokładność Walidacji: 81.12%


[Unfrozen_Low_VoG_Subset] Epoka 5/5 [Trening]:   0%|          | 0/45 [00:00<?, ?it/s]

[Unfrozen_Low_VoG_Subset] Epoka 5/5 [Walidacja]:   0%|          | 0/62 [00:00<?, ?it/s]

[Unfrozen_Low_VoG_Subset] Epoka 5/5 | Strata Treningowa: 0.0623 | Dokładność Walidacji: 86.34%

=== PODSUMOWANIE DOKŁADNOŚCI ===
Frozen_Full_Dataset: 99.57%
Frozen_High_VoG_Subset: 99.52%
Frozen_Low_VoG_Subset: 98.45%
Unfrozen_Full_Dataset: 93.61%
Unfrozen_High_VoG_Subset: 87.26%
Unfrozen_Low_VoG_Subset: 86.34%


In [ ]:
# Komórka 9: Wizualizacja
import matplotlib.pyplot as plt

def plot_training_dynamics(histories):
    epochs_range = range(1, EPOCHS_TRAIN + 1)

    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Wpływ Rankingu Danych (VoG) na Dynamikę Treningu', fontsize=18)

    colors = {'Full_Dataset': 'blue', 'High_VoG_Subset': 'red', 'Low_VoG_Subset': 'green'}

    # Rysowanie Linear Probe (Frozen)
    for key, hist in histories.items():
        if "Frozen" in key:
            ds_name = key.replace("Frozen_", "")
            axs[0, 0].plot(epochs_range, hist['train_loss'], label=ds_name, color=colors[ds_name], marker='o')
            axs[0, 1].plot(epochs_range, hist['val_acc'], label=ds_name, color=colors[ds_name], marker='o')

    axs[0, 0].set_title('Strata Treningowa - Model Zamrożony (Linear Probe)')
    axs[0, 0].set_xlabel('Epoki')
    axs[0, 0].set_ylabel('Strata (Loss)')
    axs[0, 0].legend()
    axs[0, 0].grid(True)

    axs[0, 1].set_title('Dokładność Walidacji - Model Zamrożony (Linear Probe)')
    axs[0, 1].set_xlabel('Epoki')
    axs[0, 1].set_ylabel('Dokładność (%)')
    axs[0, 1].legend()
    axs[0, 1].grid(True)

    # Rysowanie Fine-Tuning (Unfrozen)
    for key, hist in histories.items():
        if "Unfrozen" in key:
            ds_name = key.replace("Unfrozen_", "")
            axs[1, 0].plot(epochs_range, hist['train_loss'], label=ds_name, color=colors[ds_name], linestyle='--', marker='s')
            axs[1, 1].plot(epochs_range, hist['val_acc'], label=ds_name, color=colors[ds_name], linestyle='--', marker='s')

    axs[1, 0].set_title('Strata Treningowa - Model Odmrożony (Fine-Tuning)')
    axs[1, 0].set_xlabel('Epoki')
    axs[1, 0].set_ylabel('Strata (Loss)')
    axs[1, 0].legend()
    axs[1, 0].grid(True)

    axs[1, 1].set_title('Dokładność Walidacji - Model Odmrożony (Fine-Tuning)')
    axs[1, 1].set_xlabel('Epoki')
    axs[1, 1].set_ylabel('Dokładność (%)')
    axs[1, 1].legend()
    axs[1, 1].grid(True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Wywołanie funkcji generującej wykresy
plot_training_dynamics(histories)

## Podsumowanie wyników

Na pierwszym wykresie widać, że loss osiągany przez model z Linear Probe przy treningu na wszystkich obrazkach jest mniejszy od tego osiąganego na obrazach o niskim VoG, zaś ten jest mniejszy niż loss osiaągany na obrazach o wysokim VoG. Może to świadczyć o tym, że rzeczywiście modelowi trudniej jest osiągać mniejszy error na obrazach o wysokim VoG, natomiast jednak przez zbyt mały kontekst przy treningu na zbiorze o niskim VoG połączonym z ograniczonym stponiem swobody w modyfikowaniu parametrów modelu nie jest on w stanie osiągać tak dobrych wyników jak na treningu na całym zbiorze.

NA drugim wykreśie widać, że model z Linear Probe osiąga najlepsze accuracy po treningu na całym zbiorze, jednak wynik po treningu na zbiorze o wysokim VoG jest nieznacznie gorszy, a w jednej z epok sytuacja nawet uległa odwróceniu. Może to wskazywać na użyteczność metryki VoG w kontekście oszczędności mocy obliczeniowej i czasu treningu przy jednoczesnym zachowywaniu podobnej jakości modelu.

Na trzecim wykresie widzimy już trochę inną zależność. Loss dla modelu z Fine-Tuningiem trenowanym na niskim VoG osiąga mniejszą stratę niż model pełny i model o wysokim VoG. Można na tej podstawie wyciągnąć wniosek, że przy braku ograniczeń jeśli chodzi o zmiany parametrów, model oczywiście łatwiej minimallizuje stratę na obrazkach o mniejszym poziomie trudności sugerowanym przez VoG, a najgorzej mu idzie jeśli chodzi o te obrazy z wysokim VoG. 

Ostatni wykres może wskazywać na to, że chociaż model z Fine-Tuningiem trenowany na wysokim VoG może jednak być zawodny na zbiorze walidacyjnym, być może ze względu na ograniczony kontekst i w konsekwencji słabsze generalizowanie na obrazach o zróżnicowanym poziomie trudności.